# Extracción de datos - Telecom X

Este notebook extrae los datos de la API de Telecom X en formato JSON y los carga en un DataFrame de Pandas, aplanando las columnas anidadas (como `customer`, `internet`, y `account`) para facilitar el análisis.

In [ ]:
import pandas as pd
import requests
import numpy as np

In [ ]:
# URL de los datos brutos en GitHub
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

print("Descargando datos...")
response = requests.get(url)

# Verificar respuesta
if response.status_code == 200:
    data = response.json()
    
    # Aplanar el JSON y crear el DataFrame
    df = pd.json_normalize(data)
    
    print("\n¡Datos cargados con éxito!")
    print(f"Total de clientes (filas): {df.shape[0]}")
    print(f"Total de atributos (columnas): {df.shape[1]}")
else:
    print(f"Error: {response.status_code}")

In [ ]:
# Mostrar 5 primeras filas
df.head()

## 1. Exploración de la Estructura de los Datos
El primer paso fundamental es comprender qué tipos de datos tenemos y si hay valores nulos. Para esto, utilizaremos los métodos `df.info()` y `df.dtypes`.

In [ ]:
# Información General del DataFrame (Filas, Columnas, Nulos y Tipos generales)
print("--- Información del DataFrame ---")
df.info()

print("\n\n--- Tipos de Datos (dtypes) ---")
display(df.dtypes)

## Diccionario de Datos y Relevancia para la Evasión
Basado en el contexto de Telecom X, estas son las variables clave y su potencial impacto en la evasión (`Churn`):

### Variable Objetivo
- **`Churn`**: Variable categórica (Yes/No) que indica si el cliente abandonó la empresa en el último mes. **Esta es la variable más importante**, ya que es lo que queremos predecir.

### Columnas del Cliente (Demográficas)
- **`customer.tenure`**: **MUY IMPORTANTE.** Meses de permanencia en la empresa. Los clientes nuevos suelen tener tasas de evasión más altas que los clientes leales de años.
- **`customer.SeniorCitizen`**: Indica si el cliente es mayor (1) o no (0).
- **`customer.Partner` / `customer.Dependents`**: Si el cliente tiene pareja o dependientes.
- **`customer.gender`**: Género del cliente. Suele tener impacto bajo.

### Columnas de Cuenta y Cargos
- **`account.Contract`**: **MUY IMPORTANTE.** Tipo de contrato. Los contratos mes a mes casi siempre tienen el mayor índice de evasión.
- **`account.Charges.Monthly`**: Cargo mensual. **Crítico.** Aumento en costos es causa principal de abandono.
- **`account.Charges.Total`**: Cargo total histórico. Aquí debes notar en `df.info()` que fue interpretado como un objeto (string) en lugar de número, indicando necesidad de limpieza.

## 2. Verificación Continua y Limpieza
Buscar problemas en los datos que puedan afectar el análisis, como valores ausentes (NaN) o filas duplicadas.

In [ ]:
# 1. Verificar valores ausentes (Nulos reales)
nulos_totales = df.isnull().sum()
print("Valores nulos en cada columna:")
print(nulos_totales[nulos_totales > 0]) # Solo mostramos las que tienen nulos técnicos

# 2. Verificar filas duplicadas
duplicados = df.duplicated().sum()
print(f"\nNúmero de filas exactamente duplicadas: {duplicados}")

## 3. Limpieza de Errores de Formato
Hemos detectado que `account.Charges.Total` es tipo `object` debido a espacios vacíos, y que nuestra variable fundamental `Churn` también tiene registros vacíos.

In [ ]:
print("--- LIMPIEZA DE DATOS ---")

# PASO 1: Arreglar account.Charges.Total
vacios_cargos = df[df['account.Charges.Total'] == ' '].shape[0]
print(f"Cargos Totales con espacios en blanco encontrados: {vacios_cargos}")

df['account.Charges.Total'] = df['account.Charges.Total'].replace(' ', np.nan)
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'])
df['account.Charges.Total'] = df['account.Charges.Total'].fillna(0)
print("1. Columna account.Charges.Total corregida y convertida a tipo numérico.")

# PASO 2: Arreglar la variable objetivo (Churn)
churn_vacios = df[df['Churn'] == ""].shape[0]
print(f"Registros de Churn sin valor (vacíos o espacios): {churn_vacios}")

df['Churn'] = df['Churn'].replace([' ', ''], np.nan)
df = df.dropna(subset=['Churn'])
print("2. Filas con 'Churn' nulo eliminadas, ya que no sirven para entrenar el modelo.")

# Verificación final
print("\n--- RESULTADO DE LA LIMPIEZA ---")
print(f"Nuevas dimensiones: {df.shape}")

In [ ]:
# Validación de que Charges.Total ahora es float64
df.info()